# SEED2S + Reveal — South America Critical Minerals Analysis
**Region:** Latin America and the Caribbean (South America sub-region)  
**Data Source:** U.S. Census Bureau / USITC — Monthly customs import values (2010–2025)

---
## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

---
## 2. Load and Combine Data

Each CSV has 2 metadata header rows (title + export date) before the actual column headers. We skip those and combine all 4 files into one DataFrame.

In [ ]:
csv_files = [
    'Data/Raw/Copper & Graphite 3.csv',
    'Data/Raw/Lithium, Cobalt & Sliver 3.csv',
    'Data/Raw/Nickel, Manganese & Niobium 3.csv',
    'Data/Raw/Tin & Zinc 3.csv'
]

frames = []
for f in csv_files:
    df_temp = pd.read_csv(f, skiprows=2)
    frames.append(df_temp)

df = pd.concat(frames, ignore_index=True)

# Drop the trailing empty column produced by the trailing comma in the CSV
df = df.loc[:, ~df.columns.str.fullmatch('Unnamed.*')]

print(f'Combined shape: {df.shape}')
df.head()

---
## 3. Rename and Parse Columns

- Extract the 4-digit **HS Code** and the **Commodity Name** from the `Commodity` column.
- Parse the `Time` column — annual rows (e.g. `"2010"`) set Month to `NaN`; monthly rows (e.g. `"January 2010"`) extract both Month and Year.
- Rename `Customs Value (Gen) ($US)` → `Nominal Value`.

In [ ]:
import calendar

month_abbr_map = {m: calendar.month_abbr[i] for i, m in enumerate(calendar.month_name) if m}

def parse_time(t):
    parts = str(t).strip().split()
    if len(parts) == 2:
        # "January 2010" — monthly row
        return int(parts[1]), month_abbr_map.get(parts[0], np.nan)
    elif len(parts) == 1:
        # "2010" — annual summary row
        return int(parts[0]), np.nan
    return np.nan, np.nan

df[['Year', 'Month']] = df['Time'].apply(lambda t: pd.Series(parse_time(t)))

# Extract HS Code and Commodity Name
df['HS Code'] = df['Commodity'].str[:4].str.strip()
df['Commodity Name'] = df['Commodity'].str[4:].str.strip()

df.rename(columns={'Customs Value (Gen) ($US)': 'Nominal Value'}, inplace=True)

df = df[['HS Code', 'Commodity Name', 'Year', 'Month', 'Country', 'Nominal Value']]

print(df.dtypes)
print(f'\nTotal rows: {len(df)}')
print(f'Monthly rows: {df["Month"].notna().sum()}')
print(f'Annual summary rows: {df["Month"].isna().sum()}')
df.head(10)

---
## 4. Keep Monthly Rows Only

The raw data has one annual summary row per year alongside individual monthly rows. We drop the annual summaries to avoid double-counting — keeping only the monthly rows.

In [ ]:
df = df[df['Month'].notna()].reset_index(drop=True)

print(f'Rows after keeping monthly only: {len(df)}')
print('Countries in dataset:', sorted(df['Country'].unique()))

---
## 5. Filter Year Range (2010–2025)

In [ ]:
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df[(df['Year'] >= 2010) & (df['Year'] <= 2025)].reset_index(drop=True)

print(f'Year range in data: {int(df["Year"].min())} – {int(df["Year"].max())}')
print(f'Rows: {len(df)}')

---
## 6. Clean Nominal Value

Values are stored as strings with commas (e.g., `"9,941,921"`). Strip commas and convert to float.

In [6]:
df['Nominal Value'] = (
    df['Nominal Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['Nominal Value'] = pd.to_numeric(df['Nominal Value'], errors='coerce')

print(df['Nominal Value'].describe())

count    6.020000e+02
mean     1.059610e+08
std      6.066335e+08
min      2.084000e+03
25%      3.290662e+05
50%      1.856870e+06
75%      3.552581e+07
max      8.883319e+09
Name: Nominal Value, dtype: float64


---
## 7. Add Annual Inflation Rate and Calculate CPI

U.S. annual CPI inflation rates (source: U.S. Bureau of Labor Statistics).  
**Base year: 2010, Base CPI = 100**  
Formula: `CPI_current = CPI_past × (inflation_rate / 100 + 1)`

In [7]:
# U.S. annual inflation rates (%)
inflation_rates = {
    2010: 1.64,
    2011: 3.16,
    2012: 2.07,
    2013: 1.46,
    2014: 1.62,
    2015: 0.12,
    2016: 1.26,
    2017: 2.13,
    2018: 2.44,
    2019: 1.81,
    2020: 1.23,
    2021: 4.70,
    2022: 8.00,
    2023: 4.12,
    2024: 2.90,
    2025: 2.80   # estimate
}

# Calculate CPI for each year starting from base CPI = 100 in 2010
cpi_values = {}
cpi = 100.0
for year in range(2010, 2026):
    cpi_values[year] = round(cpi, 4)
    if year < 2025:
        cpi = cpi * (inflation_rates[year + 1] / 100 + 1)
    else:
        cpi = cpi * (inflation_rates[2025] / 100 + 1)

# Fix: recalculate properly — CPI for year Y uses inflation rate OF year Y applied to prior year's CPI
cpi_values = {}
cpi = 100.0
cpi_values[2010] = round(cpi, 4)
for year in range(2011, 2026):
    cpi = cpi * (inflation_rates[year] / 100 + 1)
    cpi_values[year] = round(cpi, 4)

print('CPI by year (Base 2010 = 100):')
for y, c in cpi_values.items():
    print(f'  {y}: {c}')

# Map to DataFrame
df['Annual Inflation Rate'] = df['Year'].map(inflation_rates)
df['CPI'] = df['Year'].map(cpi_values)

CPI by year (Base 2010 = 100):
  2010: 100.0
  2011: 103.16
  2012: 105.2954
  2013: 106.8327
  2014: 108.5634
  2015: 108.6937
  2016: 110.0632
  2017: 112.4076
  2018: 115.1503
  2019: 117.2345
  2020: 118.6765
  2021: 124.2543
  2022: 134.1947
  2023: 139.7235
  2024: 143.7755
  2025: 147.8012


---
## 8. Calculate Inflation-Adjusted Real Value

Formula: `Real Value = (Nominal Value / CPI) × 100`

In [8]:
df['Real Value'] = (df['Nominal Value'] / df['CPI']) * 100

print(df[['HS Code', 'Commodity Name', 'Year', 'Country', 'Nominal Value', 'CPI', 'Real Value']].head(10))

  HS Code    Commodity Name  Year    Country  Nominal Value       CPI  \
0    2504  Natural Graphite  2021  Argentina           5750  124.2543   
1    2504  Natural Graphite  2015     Brazil        9941921  108.6937   
2    2504  Natural Graphite  2016     Brazil        7426485  110.0632   
3    2504  Natural Graphite  2017     Brazil        7601671  112.4076   
4    2504  Natural Graphite  2018     Brazil        6469010  115.1503   
5    2504  Natural Graphite  2019     Brazil        5006893  117.2345   
6    2504  Natural Graphite  2020     Brazil        5464157  118.6765   
7    2504  Natural Graphite  2021     Brazil        5956447  124.2543   
8    2504  Natural Graphite  2022     Brazil        4590114  134.1947   
9    2504  Natural Graphite  2023     Brazil        4801681  139.7235   

     Real Value  
0  4.627606e+03  
1  9.146732e+06  
2  6.747473e+06  
3  6.762595e+06  
4  5.617884e+06  
5  4.270836e+06  
6  4.604245e+06  
7  4.793755e+06  
8  3.420488e+06  
9  3.436559e+06 

---
## 9. Apply Category Type Classification

Based on the Southeast Asia team's classification framework, commodities are categorized by HS code chapter:

| Category | HS Chapter / Code Range | Description |
|---|---|---|
| Ore / Raw | 25xx, 26xx | Mineral ores, earths, concentrates |
| Compound | 28xx, 7401–7402, 7501–7502, 7901–7902, 8001–8002 | Intermediate compounds, mattes, unrefined metals |
| Refined / Articles | 7403–7407, 7409–7419, 7503–7508, 7903–7907, 8003–8007 | Refined metals and simple articles |
| Advanced Product | 7408 (wire), 7419 (advanced articles), and other manufactured codes | Manufactured / value-added products |

In [9]:
def classify_hs_code(hs_code):
    """Classify HS code into category type per SEED2S+Reveal methodology."""
    try:
        code = int(hs_code)
    except (ValueError, TypeError):
        return 'Unknown'

    # Ore / Raw — mineral ores and concentrates (Chapters 25–26)
    if 2500 <= code <= 2699:
        return 'Ore / Raw'

    # Compound — intermediate / unrefined products
    if code in [2801, 2802, 2804, 2814, 2819, 2822, 2836]:  # chemical compounds
        return 'Compound'
    if 7401 <= code <= 7402:   # copper mattes, unrefined copper
        return 'Compound'
    if 7501 <= code <= 7502:   # nickel mattes, unrefined nickel
        return 'Compound'
    if 7901 <= code <= 7902:   # unwrought zinc
        return 'Compound'
    if 8001 <= code <= 8002:   # unwrought tin
        return 'Compound'

    # Advanced Product — wire, rods for further manufacture, complex articles
    if code == 7408:           # copper wire
        return 'Advanced Product'
    if code == 7505:           # nickel wire/rods
        return 'Advanced Product'
    if code == 7904:           # zinc wire/rods
        return 'Advanced Product'
    if code == 8003:           # tin bars/rods/wire
        return 'Advanced Product'

    # Refined / Articles — refined metals and simple metal articles
    if 7403 <= code <= 7419:
        return 'Refined / Articles'
    if 7503 <= code <= 7508:
        return 'Refined / Articles'
    if 7903 <= code <= 7907:
        return 'Refined / Articles'
    if 8003 <= code <= 8007:
        return 'Refined / Articles'
    if 8112 <= code <= 8113:   # niobium, germanium
        return 'Refined / Articles'

    return 'Unknown'


df['Category Type'] = df['HS Code'].apply(classify_hs_code)

print('Category distribution:')
print(df['Category Type'].value_counts())

Category distribution:
Category Type
Unknown               214
Refined / Articles    197
Compound               88
Ore / Raw              69
Advanced Product       34
Name: count, dtype: int64


---
## 10. Final Column Selection and Reorder

In [10]:
df = df[[
    'HS Code',
    'Commodity Name',
    'Year',
    'Month',
    'Country',
    'Nominal Value',
    'Annual Inflation Rate',
    'CPI',
    'Real Value',
    'Category Type'
]]

print(f'Final shape: {df.shape}')
df.head()

Final shape: (602, 10)


,HS Code,Commodity Name,Year,Month,Country,Nominal Value,Annual Inflation Rate,CPI,Real Value,Category Type
0,2504,Natural Graphite,2021,NaN,Argentina,5750,4.70,124.2543,4.627606e+03,Ore / Raw
1,2504,Natural Graphite,2015,NaN,Brazil,9941921,0.12,108.6937,9.146732e+06,Ore / Raw
2,2504,Natural Graphite,2016,NaN,Brazil,7426485,1.26,110.0632,6.747473e+06,Ore / Raw
3,2504,Natural Graphite,2017,NaN,Brazil,7601671,2.13,112.4076,6.762595e+06,Ore / Raw
4,2504,Natural Graphite,2018,NaN,Brazil,6469010,2.44,115.1503,5.617884e+06,Ore / Raw


---
## 11. Print Missing Values Per Column

In [11]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Note: Month is NaN for all rows — raw data is annual only (no monthly breakdown).')

Missing values per column:
HS Code                    0
Commodity Name             0
Year                       0
Month                    602
Country                    0
Nominal Value              0
Annual Inflation Rate      0
CPI                        0
Real Value                 0
Category Type              0
dtype: int64

Note: Month is NaN for all rows — raw data is annual only (no monthly breakdown).


---
## 12. Convert Columns to Numeric Types

In [ ]:
numeric_cols = ['Year', 'Real Value', 'Nominal Value', 'Annual Inflation Rate', 'CPI']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Year'] = df['Year'].astype('Int64')

print(df.dtypes)

---
## 13. Check for Negative Values

Checking `Year`, `Month`, `Real Value`, and `Nominal Value` for any negative entries.

In [13]:
check_cols = ['Year', 'Month', 'Real Value', 'Nominal Value']
for col in check_cols:
    neg_count = (df[col] < 0).sum()
    print(f'{col}: {neg_count} negative value(s)')

Year: 0 negative value(s)
Month: 0 negative value(s)
Real Value: 0 negative value(s)
Nominal Value: 0 negative value(s)


---
## 14. Remove Duplicate Rows

Remove rows that are identical across **all** columns.

In [14]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f'Rows before: {before} | Rows after: {after} | Duplicates removed: {before - after}')

Rows before: 602 | Rows after: 602 | Duplicates removed: 0


---
## 15. Check for Duplicate Dates per HS Code and Country

For each (HS Code, Country) pair, verify there are no repeated Year entries.

In [15]:
dupes = df[df.duplicated(subset=['HS Code', 'Country', 'Year'], keep=False)]

if dupes.empty:
    print('No duplicate (HS Code, Country, Year) combinations found.')
else:
    print(f'WARNING: {len(dupes)} rows have duplicate (HS Code, Country, Year):')
    print(dupes[['HS Code', 'Commodity Name', 'Country', 'Year', 'Nominal Value']].sort_values(
        ['HS Code', 'Country', 'Year']
    ))

No duplicate (HS Code, Country, Year) combinations found.


---
## 16. Outlier Detection

Using the IQR method to flag rows where `Real Value` is unusually high or low. **Outliers are not removed** — they are noted for awareness during analysis.

In [16]:
Q1 = df['Real Value'].quantile(0.25)
Q3 = df['Real Value'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = df[(df['Real Value'] < lower_fence) | (df['Real Value'] > upper_fence)].copy()

print(f'IQR Fences: [{lower_fence:,.0f}, {upper_fence:,.0f}]')
print(f'Outlier rows flagged: {len(outliers)}')
print()
print('Top outliers (highest Real Value):')
print(
    outliers
    .sort_values('Real Value', ascending=False)
    [['HS Code', 'Commodity Name', 'Country', 'Year', 'Nominal Value', 'Real Value']]
    .head(15)
    .to_string(index=False)
)

IQR Fences: [-42,266,290, 71,163,091]
Outlier rows flagged: 82

Top outliers (highest Real Value):
HS Code                                     Commodity Name Country  Year  Nominal Value   Real Value
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2025     8883318659 6.010316e+09
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2021     5651456655 4.548299e+09
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2024     6044245096 4.203946e+09
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2023     4584313629 3.280990e+09
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2022     4226131709 3.149254e+09
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2017     3002728546 2.671286e+09
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2018     2992780255 2.599021e+09
   7403 Refined Copper & Alloys (no Mast Alloy), Unwrought   Chile  2019     2558402873 2.182

---
## 17. Final Cleaned Dataset Summary

In [17]:
print('=== CLEANED DATASET SUMMARY ===')
print(f'Shape         : {df.shape}')
print(f'Years covered : {int(df["Year"].min())} – {int(df["Year"].max())}')
print(f'Countries     : {df["Country"].nunique()} — {sorted(df["Country"].unique())}')
print(f'HS Codes      : {df["HS Code"].nunique()} unique codes')
print(f'Commodities   : {df["Commodity Name"].nunique()} unique names')
print()
print('Category Type distribution:')
print(df['Category Type'].value_counts())
print()
print('Missing values:')
print(df.isnull().sum())
print()
df.head(10)

=== CLEANED DATASET SUMMARY ===
Shape         : (602, 10)
Years covered : 2015 – 2025
Countries     : 13 — ['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'French Guiana', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela']
HS Codes      : 29 unique codes
Commodities   : 29 unique names

Category Type distribution:
Category Type
Unknown               214
Refined / Articles    197
Compound               88
Ore / Raw              69
Advanced Product       34
Name: count, dtype: int64

Missing values:
HS Code                    0
Commodity Name             0
Year                       0
Month                    602
Country                    0
Nominal Value              0
Annual Inflation Rate      0
CPI                        0
Real Value                 0
Category Type              0
dtype: int64



,HS Code,Commodity Name,Year,Month,Country,Nominal Value,Annual Inflation Rate,CPI,Real Value,Category Type
0,2504,Natural Graphite,2021,NaN,Argentina,5750,4.70,124.2543,4.627606e+03,Ore / Raw
1,2504,Natural Graphite,2015,NaN,Brazil,9941921,0.12,108.6937,9.146732e+06,Ore / Raw
2,2504,Natural Graphite,2016,NaN,Brazil,7426485,1.26,110.0632,6.747473e+06,Ore / Raw
3,2504,Natural Graphite,2017,NaN,Brazil,7601671,2.13,112.4076,6.762595e+06,Ore / Raw
4,2504,Natural Graphite,2018,NaN,Brazil,6469010,2.44,115.1503,5.617884e+06,Ore / Raw
5,2504,Natural Graphite,2019,NaN,Brazil,5006893,1.81,117.2345,4.270836e+06,Ore / Raw
6,2504,Natural Graphite,2020,NaN,Brazil,5464157,1.23,118.6765,4.604245e+06,Ore / Raw
7,2504,Natural Graphite,2021,NaN,Brazil,5956447,4.70,124.2543,4.793755e+06,Ore / Raw
8,2504,Natural Graphite,2022,NaN,Brazil,4590114,8.00,134.1947,3.420488e+06,Ore / Raw
9,2504,Natural Graphite,2023,NaN,Brazil,4801681,4.12,139.7235,3.436559e+06,Ore / Raw


---
## 18. Export Cleaned Data to CSV

In [ ]:
df.to_csv('Data/Processed/south_america_cleaned.csv', index=False)
print('Saved: Data/Processed/south_america_cleaned.csv')